# Mini-projet : Planification robuste sur grille — A* + Chaînes de Markov

**ENSET Mohammadia · 2025–2026**

---
**Comment utiliser ce notebook :**
- Exécutez **Kernel → Restart & Run All** pour tout relancer proprement.
- Chaque section est **autonome** : elle redéfinit les variables dont elle a besoin.
- Les modules `astar.py`, `markov.py` et `experiments.py` doivent être dans le **même dossier**.

## 0. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from astar import (
    astar, ucs, greedy, astar_weighted,
    path_to_policy, print_grid,
    heuristic_manhattan, heuristic_zero, heuristic_euclidean,
    best_first_search
)
from markov import (
    build_transition_matrix,
    distribution_trajectory,
    communication_classes,
    absorption_analysis,
    simulate_trajectories,
    GOAL_STATE, FAIL_STATE
)
from experiments import (
    make_grid_simple, make_grid_maze, make_grid_weighted,
    experiment_algo_comparison, experiment_epsilon_impact,
    experiment_markov_analysis, experiment_heuristic_comparison,
    visualize_grid_path, _plot_grid
)

print('✅ Tous les modules chargés avec succès.')


✅ Tous les modules chargés avec succès.


---
## 1. Définition de la grille
Modifiez la ligne `grid = make_grid_simple()` pour changer de grille :
- `make_grid_simple()` — grille 8×8 avec obstacles
- `make_grid_maze()` — labyrinthe 10×10
- `make_grid_weighted()` — grille 8×8 avec coûts variables

In [2]:
# ── Choisir la grille ici ────────────────────────────────────────────────
grid  = make_grid_simple()   # ← modifier : make_grid_maze() ou make_grid_weighted()
start = (0, 0)
goal  = (7, 7)               # ← (9,9) pour le labyrinthe
EPSILON = 0.1
# ─────────────────────────────────────────────────────────────────────────

print(f'Grille {len(grid)}×{len(grid[0])}  |  Start={start}  |  Goal={goal}  |  ε={EPSILON}')
print()
print_grid(grid, start=start, goal=goal)

Grille 8×8  |  Start=(0, 0)  |  Goal=(7, 7)  |  ε=0.1

S               
    #           
    #           
    #   # # #   
    #           
  # #           
            #   
            # G 


---
## 2. Planification déterministe — Comparaison UCS / Greedy / A* / A* pondéré

Les 4 algorithmes sont lancés. Le tableau comparatif et les 4 chemins sont affichés.
> **Rappel :** A* avec heuristique admissible (Manhattan) garantit l'optimalité.

In [3]:
# ── Comparaison des 4 algorithmes ──────────────────────────────────────
algos = experiment_algo_comparison(grid, start, goal, name='notebook')

# ── Affichage du chemin A* ──────────────────────────────────────────────
res_astar = algos['A*']
print()
print_grid(grid, path=res_astar['path'], start=start, goal=goal)
print(f'\nChemin A* : {len(res_astar["path"])} états, coût = {res_astar["cost"]}')
print(f'Chemin    : {res_astar["path"]}')


  Comparaison algorithmes — notebook
  Grille 8x8, start=(0, 0), goal=(7, 7)
Algorithme               Coût   Nœuds exp.   OPEN max  Temps(ms)
-----------------------------------------------------------------
UCS                     14.00           53          7      0.219
Greedy                  14.00           15         10      0.061
A*                      14.00           53          7      0.178
A* pondéré(w=2)         14.00           15         10      0.049
  → Figure sauvegardée : notebook_algo_comparison.png

S . . . . . . . 
    #         . 
    #         . 
    #   # # # . 
    #         . 
  # #         . 
            # . 
            # G 

Chemin A* : 15 états, coût = 14.0
Chemin    : [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7)]


In [5]:
# ── Visualisation côte à côte des 4 chemins ────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f'Comparaison des algorithmes — {len(grid)}×{len(grid[0])}', fontsize=13, fontweight='bold')
for ax, (name, res) in zip(axes, algos.items()):
    _plot_grid(ax, grid, res.get('path', []), start, goal,
               f'{name}\nCoût={res["cost"]:.1f} | Nœuds={res["nodes_exp"]}', res)
plt.tight_layout()
plt.show()

C:\Users\pc\AppData\Local\Temp\ipykernel_16728\1154666370.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Construction de la matrice de Markov P

La **politique** est extraite du chemin A*. La matrice **P** est construite avec le paramètre ε.

Modèle d'incertitude :
- action principale réussie avec proba `1 − ε`
- déviation gauche ou droite avec proba `ε/2` chacune
- collision / hors-grille → self-loop (l'agent reste sur place)

In [7]:
# ── Politique du chemin A* ──────────────────────────────────────────────
res_astar = algos['A*']
policy    = path_to_policy(res_astar['path'])

# ── Construction de P ──────────────────────────────────────────────────
P, idx = build_transition_matrix(policy, grid, goal, epsilon=EPSILON)
state_to_i = {s: i for i, s in enumerate(idx)}

print(f'Dimension de P    : {P.shape}')
print(f'P stochastique    : {np.allclose(P.sum(axis=1), 1.0)}')
print(f'GOAL absorbant    : {P[state_to_i[GOAL_STATE], state_to_i[GOAL_STATE]] == 1.0}')
print(f'\nIndex des états :')
for i, s in enumerate(idx):
    print(f'  [{i:2d}] {s}')

Dimension de P    : (15, 15)
P stochastique    : True
GOAL absorbant    : True

Index des états :
  [ 0] (0, 0)
  [ 1] (0, 1)
  [ 2] (0, 2)
  [ 3] (0, 3)
  [ 4] (0, 4)
  [ 5] (0, 5)
  [ 6] (0, 6)
  [ 7] (0, 7)
  [ 8] (1, 7)
  [ 9] (2, 7)
  [10] (3, 7)
  [11] (4, 7)
  [12] (5, 7)
  [13] (6, 7)
  [14] GOAL


In [8]:
# ── Heatmap de la matrice P ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(P, cmap='Reds', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
labels = [str(s) for s in idx]
ax.set_xticks(range(len(idx))); ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(idx))); ax.set_yticklabels(labels, fontsize=8)
ax.set_title(f'Matrice de transition P  (ε = {EPSILON})', fontsize=12)
plt.tight_layout()
plt.show()

C:\Users\pc\AppData\Local\Temp\ipykernel_16728\3755276606.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Analyse de la chaîne de Markov

### 4.1 Classes de communication (algorithme de Kosaraju)
### 4.2 Analyse d'absorption : N = (I−Q)⁻¹, B = N·R, t = N·1
### 4.3 Évolution de la distribution π⁽ⁿ⁾ = π⁽⁰⁾·Pⁿ

In [9]:
# ── Sécurité : reconstruire P si cellule exécutée isolément ─────────────
if 'P' not in vars() or 'idx' not in vars():
    res_astar = algos['A*']
    policy    = path_to_policy(res_astar['path'])
    P, idx    = build_transition_matrix(policy, grid, goal, epsilon=EPSILON)
    state_to_i = {s: i for i, s in enumerate(idx)}

# ── 4.1 Classes de communication ────────────────────────────────────────
cc = communication_classes(P, idx)
print(f'Nombre de classes (SCC) : {cc["n_scc"]}')
rec   = sum(1 for t in cc['type'].values() if t == 'recurrent')
trans = sum(1 for t in cc['type'].values() if t == 'transient')
print(f'États récurrents        : {rec}  |  États transitoires : {trans}')
print()
for s, t in cc['type'].items():
    print(f'  {str(s):25s}  → {t}')

Nombre de classes (SCC) : 14
États récurrents        : 1  |  États transitoires : 14

  (0, 0)                     → transient
  (0, 1)                     → transient
  (0, 2)                     → transient
  (0, 3)                     → transient
  (0, 4)                     → transient
  (0, 5)                     → transient
  (0, 6)                     → transient
  (0, 7)                     → transient
  (1, 7)                     → transient
  (2, 7)                     → transient
  (3, 7)                     → transient
  (4, 7)                     → transient
  (5, 7)                     → transient
  (6, 7)                     → transient
  GOAL                       → recurrent


In [10]:
# ── 4.2 Analyse d'absorption ────────────────────────────────────────────
absorb = absorption_analysis(P, idx)
if absorb:
    print('=== Analyse d\'absorption ===')
    print(f'États absorbants : {absorb["absorbing_states"]}')
    print(f'\nTemps moyen d\'absorption E[T] depuis chaque état :')
    for s, t in absorb['t_mean'].items():
        print(f'  {str(s):25s}  E[T] = {t:.2f} pas')
    b = absorb['B'].get(start, {}).get(GOAL_STATE, None)
    t_start = absorb['t_mean'].get(start, None)
    if b is not None:
        print(f'\nP(GOAL | départ={start}) = {b:.4f}')
    if t_start is not None:
        print(f'E[T     | départ={start}] = {t_start:.2f} pas')

=== Analyse d'absorption ===
États absorbants : ['GOAL']

Temps moyen d'absorption E[T] depuis chaque état :
  (0, 0)                     E[T] = 15.62 pas
  (0, 1)                     E[T] = 14.51 pas
  (0, 2)                     E[T] = 13.40 pas
  (0, 3)                     E[T] = 12.28 pas
  (0, 4)                     E[T] = 11.17 pas
  (0, 5)                     E[T] = 10.06 pas
  (0, 6)                     E[T] = 8.95 pas
  (0, 7)                     E[T] = 7.84 pas
  (1, 7)                     E[T] = 6.67 pas
  (2, 7)                     E[T] = 5.56 pas
  (3, 7)                     E[T] = 4.44 pas
  (4, 7)                     E[T] = 3.33 pas
  (5, 7)                     E[T] = 2.22 pas
  (6, 7)                     E[T] = 1.11 pas

P(GOAL | départ=(0, 0)) = 1.0000
E[T     | départ=(0, 0)] = 15.62 pas


In [11]:
# ── 4.3 Distribution π⁽ⁿ⁾ ──────────────────────────────────────────────
n_steps = 60
pi0     = np.zeros(len(idx))
pi0[state_to_i[start]] = 1.0

traj   = distribution_trajectory(pi0, P, n_steps)
goal_i = state_to_i[GOAL_STATE]

plt.figure(figsize=(9, 4))
plt.plot(range(n_steps + 1), traj[:, goal_i], color='#C0392B', lw=2.5)
plt.xlabel('Pas de temps n', fontsize=11)
plt.ylabel('π⁽ⁿ⁾[GOAL]', fontsize=11)
plt.title(f'Probabilité d\'être dans GOAL à l\'instant n  (ε = {EPSILON})', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'π⁽⁵⁰⁾[GOAL] = {traj[50, goal_i]:.4f}')
print(f'π⁽⁶⁰⁾[GOAL] = {traj[60, goal_i]:.4f}')

π⁽⁵⁰⁾[GOAL] = 1.0000
π⁽⁶⁰⁾[GOAL] = 1.0000


C:\Users\pc\AppData\Local\Temp\ipykernel_16728\2561352356.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Simulation Monte-Carlo

5 000 trajectoires simulées depuis `start`. Comparaison P(GOAL) et E[T] empiriques vs théoriques.

> **Note :** P(GOAL) = 1 dans ce modèle car P est construite uniquement sur les états du chemin.
> Toute déviation hors chemin produit un self-loop. Un modèle complet sur toute la grille
> donnerait P(GOAL) < 1 pour ε > 0.

In [12]:
# ── Sécurité ────────────────────────────────────────────────────────────
if 'P' not in vars() or 'idx' not in vars():
    res_astar  = algos['A*']
    policy     = path_to_policy(res_astar['path'])
    P, idx     = build_transition_matrix(policy, grid, goal, epsilon=EPSILON)
    state_to_i = {s: i for i, s in enumerate(idx)}
if 'absorb' not in vars():
    absorb = absorption_analysis(P, idx)

# ── Simulation ──────────────────────────────────────────────────────────
sim = simulate_trajectories(P, idx, start, n_simulations=5000, max_steps=300)

print(f'Résultats Monte-Carlo (N = {sim["n_simulations"]}) :')
print(f'  P̂(GOAL)     = {sim["prob_goal"]:.4f}')
print(f'  P̂(timeout)  = {sim["prob_timeout"]:.4f}')
print(f'  Ê[T|GOAL]   = {sim["mean_time_goal"]:.2f} ± {sim["std_time_goal"]:.2f} pas')
if absorb:
    t_theo = absorb['t_mean'].get(start, None)
    b_theo = absorb['B'].get(start, {}).get(GOAL_STATE, None)
    if t_theo: print(f'  E[T] théo   = {t_theo:.2f} pas  (écart MC = {abs(sim["mean_time_goal"]-t_theo):.2f})')
    if b_theo: print(f'  P(GOAL) th  = {b_theo:.4f}')

Résultats Monte-Carlo (N = 5000) :
  P̂(GOAL)     = 1.0000
  P̂(timeout)  = 0.0000
  Ê[T|GOAL]   = 15.59 ± 1.39 pas
  E[T] théo   = 15.62 pas  (écart MC = 0.02)
  P(GOAL) th  = 1.0000


In [18]:
# ── Courbe π(n) matricielle vs CDF empirique + histogramme ─────────────
if 'traj' not in vars():
    pi0  = np.zeros(len(idx)); pi0[state_to_i[start]] = 1.0
    traj = distribution_trajectory(pi0, P, 60)

goal_i = state_to_i[GOAL_STATE]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Analyse Markov  (ε = {EPSILON})', fontsize=12, fontweight='bold')

# Gauche : convergence
axes[0].plot(range(len(traj)), traj[:, goal_i], color='#C0392B', lw=2.5, label='Matriciel P^n')
if sim['times_goal']:
    ts = sorted(sim['times_goal'])
    ecdf = np.arange(1, len(ts)+1) / sim['n_simulations']
    axes[0].step(ts, ecdf, color='#8B0000', lw=1.5, alpha=0.7, label='Monte-Carlo CDF')
axes[0].set_xlabel('Pas n'); axes[0].set_ylabel('P(Xn = GOAL)')
axes[0].set_title('Convergence vers GOAL'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Droite : histogramme
if sim['times_goal']:
    axes[1].hist(sim['times_goal'], bins=30, color='#8B0000', alpha=0.75, edgecolor='white')
    axes[1].axvline(sim['mean_time_goal'], color='red', lw=2, ls='--',
                    label=f'MC E[T] = {sim["mean_time_goal"]:.1f}')
    if absorb and absorb['t_mean'].get(start):
        axes[1].axvline(absorb['t_mean'][start], color='#2C3E50', lw=2, ls=':',
                        label=f'Théo E[T] = {absorb["t_mean"][start]:.1f}')
    axes[1].set_xlabel('Temps d\'atteinte (pas)'); axes[1].set_ylabel('Fréquence')
    axes[1].set_title('Distribution du temps d\'atteinte GOAL'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\pc\AppData\Local\Temp\ipykernel_16728\3367399842.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Impact de ε — Analyse de robustesse (E.2)

On fait varier l'incertitude **ε ∈ {0.00, 0.10, 0.20, 0.30}** (conformément à E.2) et on mesure :
- **(i)** le coût planifié par A* (déterministe, constant)
- **(ii)** la probabilité réelle P(GOAL) via la chaîne de Markov
- **(iii)** le temps moyen d'atteinte E[T|GOAL]

> **Limite du modèle :** P(GOAL) = 1 pour tout ε car P est restreinte aux états du chemin A*.
> Le coût réel de l'incertitude est un **allongement du trajet** : +45% de ε=0 à ε=0.30.


In [19]:
# ── E.2 : A* fixé, variation de ε ∈ {0, 0.1, 0.2, 0.3} ───────────────
results_eps = experiment_epsilon_impact(
    grid, start, goal,
    epsilons=[0.0, 0.1, 0.2, 0.3],   # valeurs E.2 exactes
    n_sim=3000,
    name='notebook'
)
plt.show()



  E.2 Impact de ε — notebook
  Coût A* (déterministe) : 14.0  |  15 étapes

      ε |  Coût A* |  P(GOAL) |  E[T|GOAL] |  Timeout
  -------------------------------------------------------
   0.00 |     14.0 |    1.000 |       14.0 |    0.000
   0.10 |     14.0 |    1.000 |       15.6 |    0.000
   0.20 |     14.0 |    1.000 |       17.6 |    0.000
   0.30 |     14.0 |    1.000 |       20.3 |    0.000
  → Figure sauvegardée : notebook_epsilon_impact.png


C:\Users\pc\AppData\Local\Temp\ipykernel_16728\2984211765.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. Comparaison des heuristiques admissibles (E.3)

Trois heuristiques admissibles sont comparées : **h=0** (UCS), **h=Manhattan**, **h=Euclidean**.
Toutes garantissent l'optimalité — la différence est l'**efficacité** (nœuds développés).

| Heuristique | Admissible | Informativité |
|---|---|---|
| h = 0 | Oui (triviale) | Nulle — aucun guidage |
| h = Manhattan | Oui | Forte — exacte sans obstacles |
| h = Euclidean | Oui | Moyenne — sous-estime plus que Manhattan en 4-connexité |


In [15]:
# ── E.3 : Comparaison heuristiques admissibles ──────────────────────────
heuristic_results = experiment_heuristic_comparison(grid, start, goal, name='notebook')
plt.show()



  E.3 Comparaison heuristiques — notebook
  Grille 8x8, start=(0, 0), goal=(7, 7)

  Heuristique             Admissible     Coût   Nœuds dév.   OPEN max  Temps(ms)
  ------------------------------------------------------------------------------
  h = 0  (UCS)           Oui (triviale)    14.00           53          7      0.271
  h = Manhattan (A*)             Oui    14.00           53          7      0.278
  h = Euclidean (A*)             Oui    14.00           53         11      0.311
  → Figures sauvegardées : notebook_heuristic_paths.png / notebook_heuristic_bars.png


---
## 7. Récapitulatif des résultats

In [16]:
print('=' * 60)
print('RÉSUMÉ DES RÉSULTATS CLÉS')
print('=' * 60)

# ── Planification ────────────────────────────────────────────────────────
r_ucs = algos['UCS']
r_ast = algos['A*']
print(f'\n[Planification — grille {len(grid)}×{len(grid[0])}]')
print(f'  Coût A* = {r_ast["cost"]:.1f}  (identique à UCS = {r_ucs["cost"]:.1f})')
if r_ucs['nodes_exp'] > 0:
    red = (1 - r_ast['nodes_exp'] / r_ucs['nodes_exp']) * 100
    print(f'  Réduction nœuds A* vs UCS : {red:.1f}%')

# ── Markov ───────────────────────────────────────────────────────────────
if 'sim' in vars() and sim:
    print(f'\n[Markov — ε = {EPSILON}]')
    print(f'  P̂(GOAL) Monte-Carlo  = {sim["prob_goal"]:.4f}')
    if absorb:
        b = absorb['B'].get(start, {}).get(GOAL_STATE)
        t = absorb['t_mean'].get(start)
        if b: print(f'  P(GOAL) analytique   = {b:.4f}')
        if t: print(f'  E[T]   analytique    = {t:.2f} pas')
    print(f'  Ê[T]   Monte-Carlo   = {sim["mean_time_goal"]:.2f} pas')
    if absorb and absorb['t_mean'].get(start):
        ecart = abs(sim['mean_time_goal'] - absorb['t_mean'][start])
        print(f'  Écart théo/MC        = {ecart:.3f} pas ({ecart/absorb["t_mean"][start]*100:.2f}%)')

# ── Impact ε ─────────────────────────────────────────────────────────────
if 'results_eps' in vars() and results_eps:
    print(f'\n[Impact de ε sur E[T|GOAL] (modèle chemin-restreint)]')
    eps0 = list(results_eps.keys())[0]
    et0  = results_eps[eps0]['mean_time_goal'] or 0
    for eps, r in results_eps.items():
        et  = r['mean_time_goal'] or 0
        aug = (et - et0) / et0 * 100 if et0 else 0
        print(f'  ε={eps:.2f} → E[T]={et:.1f} pas  (+{aug:.0f}%)')

print()
print('Note : P(GOAL)=1 pour tout ε est un artefact du modèle (P restreinte au chemin A*).')
print('       Cf. section 8.2 du rapport pour la discussion complète.')

RÉSUMÉ DES RÉSULTATS CLÉS

[Planification — grille 8×8]
  Coût A* = 14.0  (identique à UCS = 14.0)
  Réduction nœuds A* vs UCS : 0.0%

[Markov — ε = 0.1]
  P̂(GOAL) Monte-Carlo  = 1.0000
  P(GOAL) analytique   = 1.0000
  E[T]   analytique    = 15.62 pas
  Ê[T]   Monte-Carlo   = 15.59 pas
  Écart théo/MC        = 0.022 pas (0.14%)

[Impact de ε sur E[T|GOAL] (modèle chemin-restreint)]
  ε=0.00 → E[T]=14.0 pas  (+0%)
  ε=0.10 → E[T]=15.6 pas  (+11%)
  ε=0.20 → E[T]=17.6 pas  (+26%)
  ε=0.30 → E[T]=20.3 pas  (+45%)

Note : P(GOAL)=1 pour tout ε est un artefact du modèle (P restreinte au chemin A*).
       Cf. section 8.2 du rapport pour la discussion complète.
